# 02. 이미지 다운로드

`data/metadata/manifest.csv`의 OriginalURL에서 이미지를 `data/raw/`에 다운로드합니다.

**전제조건**: `01_download_metadata.ipynb` 실행 완료

**특징**
- 이미 다운로드된 파일은 자동 스킵 (재실행 안전)
- `config.MAX_WORKERS` 스레드로 병렬 다운로드
- 실패한 이미지는 출력에 표시되고 건너뜀

In [1]:
import os
import sys
import requests
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import config

manifest = pd.read_csv(config.MANIFEST_PATH)
print(f'다운로드 대상 이미지 수 (unique): {manifest["ImageID"].nunique()}')
print(manifest['ClassName'].value_counts())

다운로드 대상 이미지 수 (unique): 174
ClassName
Refrigerator       100
Washing machine    100
Name: count, dtype: int64


In [2]:
def download_image(row):
    """단일 이미지 다운로드. 반환값: (status, detail)"""
    image_id = row['ImageID']
    url      = row['OriginalURL']
    save_path = os.path.join(config.RAW_DIR, f'{image_id}.jpg')

    if os.path.exists(save_path):
        return 'skip', image_id

    try:
        r = requests.get(url, timeout=config.DOWNLOAD_TIMEOUT)
        r.raise_for_status()
        with open(save_path, 'wb') as f:
            f.write(r.content)
        return 'ok', image_id
    except Exception as e:
        return 'fail', f'{image_id}: {e}'

In [3]:
# 중복 이미지 제거 (bbox가 여러 개여도 이미지는 한 번만 다운로드)
unique_rows = manifest.drop_duplicates(subset=['ImageID']).to_dict('records')
results = {'ok': 0, 'skip': 0, 'fail': 0}
fail_log = []

with ThreadPoolExecutor(max_workers=config.MAX_WORKERS) as executor:
    futures = {executor.submit(download_image, row): row for row in unique_rows}
    with tqdm(total=len(futures), desc='이미지 다운로드') as bar:
        for future in as_completed(futures):
            status, detail = future.result()
            results[status] += 1
            if status == 'fail':
                fail_log.append(detail)
            bar.set_postfix(results)
            bar.update(1)

print(f'\n완료 — 성공: {results["ok"]}, 스킵: {results["skip"]}, 실패: {results["fail"]}')
if fail_log:
    print('\n실패 목록:')
    for msg in fail_log:
        print(f'  {msg}')

이미지 다운로드:   0%|          | 0/174 [00:00<?, ?it/s]


완료 — 성공: 146, 스킵: 0, 실패: 28

실패 목록:
  3fe2b1f65f3238f9: 404 Client Error: Not Found for url: https://c2.staticflickr.com/9/8249/8655831436_bb728569bd_o.jpg
  701eaffd38f3bdcf: 404 Client Error: Not Found for url: https://c8.staticflickr.com/7/6188/6086574590_d934a5d74b_o.jpg
  07c7da6ee633bac1: 404 Client Error: Not Found for url: https://farm2.staticflickr.com/71/212161152_e2d961d73a_o.jpg
  012570c9ec60f9c4: 404 Client Error: Not Found for url: https://farm5.staticflickr.com/5094/5454796475_b66e70486e_o.jpg
  98b79845429b0080: 404 Client Error: Not Found for url: https://farm1.staticflickr.com/2480/3758712859_7d985eb567_o.jpg
  81d653f8cc985f18: 404 Client Error: Not Found for url: https://c1.staticflickr.com/4/3649/3621273966_7079f8c555_o.jpg
  99e6b2e3c3cfda19: 404 Client Error: Not Found for url: https://c2.staticflickr.com/8/7088/7169701719_3dd1531b1f_o.jpg
  1458ece265475557: 404 Client Error: Not Found for url: https://farm8.staticflickr.com/3130/3149877408_a551337c8e_o.jpg
  

In [4]:
# 다운로드 결과 검증
downloaded = [f for f in os.listdir(config.RAW_DIR) if f.endswith('.jpg')]
print(f'data/raw/ 파일 수: {len(downloaded)}')

# manifest에 있는 이미지 중 실제로 다운로드된 것만 확인
expected_ids = set(manifest['ImageID'].unique())
downloaded_ids = {os.path.splitext(f)[0] for f in downloaded}
missing = expected_ids - downloaded_ids
if missing:
    print(f'[경고] manifest에 있지만 파일 없음: {len(missing)}개')
    print('  → 해당 이미지는 03_crop_bbox.ipynb에서 자동 스킵됩니다.')
else:
    print('모든 이미지 다운로드 확인 완료')

print('\n다음 단계: 03_crop_bbox.ipynb 실행')

data/raw/ 파일 수: 146
[경고] manifest에 있지만 파일 없음: 28개
  → 해당 이미지는 03_crop_bbox.ipynb에서 자동 스킵됩니다.

다음 단계: 03_crop_bbox.ipynb 실행
